In [340]:
import pandas as pd
import requests
from urllib.robotparser import RobotFileParser
from urllib.parse import urlsplit
from pathlib import Path
import time
import re
from datetime import datetime, timezone
import json

from bs4 import BeautifulSoup

In [329]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [330]:
# BASE = Path(__file__).resolve().parent.parent
# RAW = BASE / "data" / "raw"

In [333]:
AGENT = "INFNET-TRABALHO-ACADEMICO-CIENCIA DE DADOS"
HEADER = {"User-Agent": AGENT}
DELAY = 10.0
TIME_LIMIT = 30

SITES = [
    "https://g1.globo.com/educacao",
    "https://agenciabrasil.ebc.com.br/educacao",
    # "https://educacao.uol.com.br",
    # 'https://brasilescola.uol.com.br',
    # 'https://vestibular.brasilescola.uol.com.br',
    # 'https://educador.brasilescola.uol.com.br'

]

In [344]:
def permissao(url):
  rp = RobotFileParser()
  robot = requests.get(
      f'{url}/robots.txt',
      headers=HEADER,
      timeout=TIME_LIMIT,
  ).text
  rp.parse(robot.splitlines())
  delay = rp.crawl_delay(AGENT)
  if delay is None:
    delay = DELAY
  return rp, delay



def scraping(url, rp, delay):
  if not rp.can_fetch(AGENT, url):
    raise PermissionError(f"robots.txt não permite: {url}")

  resposta = requests.get(url, headers=HEADER, timeout=TIME_LIMIT)
  resposta.raise_for_status()
  resposta.encoding = "utf-8"

  time.sleep(delay)
  return resposta.text



def noticias(html):
  sopa = BeautifulSoup(html, "html.parser")
  titulo = sopa.title.text

  lista_links = []
  links = sopa.find_all("a", href=True)

  if 'Agência Brasil' in titulo:
    host = 'Agência Brasil - EBC'
    url_ebc = 'https://agenciabrasil.ebc.com.br'
    for link in links:
      endereco = link["href"]
      filtro = ("/noticia/" not in endereco)
      if filtro:
        continue
      link = f'{url_ebc}{link["href"]}'
      if link not in lista_links:
        lista_links.append(link)

  elif 'G1 Educação' in titulo:
    host = 'g1 - Portal de notícias da Globo'
    for link in links:
      endereco = link["href"]
      filtro = ("/noticia/" not in endereco) or ("g1.globo.com/educacao/" not in endereco)
      if filtro:
        continue
      if endereco not in lista_links:
        lista_links.append(endereco)

# elif 'UOL' in titulo:
#   host = 'uol'
#   for link in links:
#     endereco = link["href"]
#     filtro = ("/noticias/" not in endereco) or (("brasilescola.uol" not in endereco) and ('educacao.uol' not in endereco))
#     if filtro:
#       continue
#     if endereco not in lista_links:
#       lista_links.append(endereco)
#   return lista_links

  else:
    print("Site não identificado")
    return None

  return lista_links, host



def get_materia(link):
    partes = urlsplit(link)
    base = f"{partes.scheme}://{partes.hostname}"
    arquivo = partes.path.split("/")[-1]

    try:
        rp, delay = permissao(base)
        html = scraping(link, rp, delay)

    except PermissionError as erro:
        print(f'ERRO: {link} - {erro}')
        return None

    return html


# BASE = Path(__file__).resolve().parent.parent
# RAW = BASE / "data" / "raw"


def gravar(linhas):
  pasta = Path(
    "/content/drive/MyDrive/"
    "Colab Notebooks/"
    "Bloco 5/"
    "Projeto de Bloco: Inteligência Artificial Aplicada [26E3_5]"
    )
  agora = datetime.now(timezone.utc).astimezone()

  data_arquivo = agora.strftime("%d_%m_%Y")
  coletado = agora.strftime("%d/%m/%Y %H:%M")


  caminho_csv = pasta / f"noticias_{data_arquivo}.csv"
  df = pd.DataFrame(linhas)
  df.to_csv(
      caminho_csv,
      index=False,
      encoding="utf-8"
  )

  # Metadados
  metadados = {
      "coletado_em": coletado,
      "sites": SITES,
      "noticias": len(linhas)
  }

  caminho_json = pasta / f"noticias_{data_arquivo}.json"

  with open(caminho_json, "w", encoding="utf-8") as arquivo:
      json.dump(
          metadados,
          arquivo,
          ensure_ascii=False,
          indent=4
      )




def coletar():
  linhas = []

  for site in SITES:
    rp, delay = permissao(site)
    html = scraping(site, rp, delay)

    lista_links, host = noticias(html)
    if lista_links is None:
      continue

    for link in lista_links:
      html = get_materia(link)

      if html is None:
        continue

      sopa = BeautifulSoup(html, "html.parser")

      if host == 'Agência Brasil - EBC':
        titulo = sopa.find("h1")
        if titulo is None:
            continue
        titulo = titulo.get_text(" ", strip=True)

        data = sopa.find("div", class_="data").text.strip()
        data = re.search(r"(\d{2}/\d{2}/\d{4})\s*-\s*(\d{2}:\d{2})", data).group()

        paragrafos = []
        for i in sopa.find("div", class_="conteudo-noticia").find_all("p"):
          if len(i.get_text(' ', strip=True)) > 10:
            paragrafos.append(i.get_text(' ', strip=True))
        texto = "\n".join(paragrafos)
        palavras = len(texto.split())

      if host == 'g1 - Portal de notícias da Globo':
        titulo = sopa.find("h1")
        if titulo is None:
            continue
        titulo = titulo.get_text(" ", strip=True)

        data = sopa.find("p", class_="content-publication-data__updated").text.strip()
        data = re.search(r"(\d{2}/\d{2}/\d{4})\s*(\d{2}h\d{2})", data).group().replace('h', ':').replace(' ', ' - ')

        paragrafos = []
        paragrafos.append(sopa.find("h2", class_='content-head__subtitle').text.strip())

        for i in sopa.find_all("p", class_= 'content-text__container'):
          if len(i.get_text(' ', strip=True)) > 10:
            paragrafos.append(i.get_text(' ', strip=True))
        texto = "\n".join(paragrafos)
        palavras = len(texto.split())

      linha = {
          "titulo": titulo,
          "data": data,
          "texto": texto,
          "palavras": palavras,
          "url": link,
          "fonte": host
      }
      linhas.append(linha)
  gravar(linhas)

In [345]:
inicio = time.perf_counter()
coletar()
tempo_coleta = time.perf_counter() - inicio
print(f"Tempo de coleta: {tempo_coleta:.2f} segundos.")

Tempo de coleta: 278.37 segundos.


In [328]:
pasta = Path.cwd()

print(f'{pasta}/')

/content


In [141]:
for site in SITES:
    try:
        rp, delay = permissao(site)

        if not rp.can_fetch(AGENT, site):
            raise PermissionError(f"robots.txt não permite: {site}")

        print(f"PERMITIDO: {site} | delay = {delay}s")

    except PermissionError as erro:
        print(f"NAO PERMITIDO: {site} - {erro}")

PERMITIDO: https://g1.globo.com/educacao | delay = 10.0s
PERMITIDO: https://agenciabrasil.ebc.com.br/educacao | delay = 10.0s
PERMITIDO: https://educacao.uol.com.br | delay = 10.0s
PERMITIDO: https://brasilescola.uol.com.br | delay = 10.0s
PERMITIDO: https://vestibular.brasilescola.uol.com.br | delay = 10.0s
PERMITIDO: https://educador.brasilescola.uol.com.br | delay = 10.0s


# **testes**

---

teste_ebc

In [282]:
rp, delay = permissao('https://agenciabrasil.ebc.com.br/educacao')

In [283]:
html = scraping('https://agenciabrasil.ebc.com.br/educacao', rp, delay)

In [285]:
html

'<!DOCTYPE html>\n<html lang="pt-br" dir="ltr"\n  xmlns:fb="http://ogp.me/ns/fb#"\n  xmlns:og="http://ogp.me/ns#">\n\n<head>\n  <meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\n<link rel="shortcut icon" href="https://agenciabrasil.ebc.com.br/sites/default/themes/agenciabrasil_v3/favicon.ico?thzdkc" />\n<link rel="canonical" href="https://agenciabrasil.ebc.com.br/educacao" />\n<meta property="og:site_name" content="Agência Brasil" />\n<meta property="og:type" content="article" />\n<meta property="og:url" content="https://agenciabrasil.ebc.com.br/educacao" />\n<meta property="og:title" content="Educação" />\n<meta name="twitter:card" content="summary" />\n<meta name="twitter:site" content="@agenciabrasil" />\n<meta name="twitter:creator" content="@agenciabrasil" />\n<meta name="twitter:url" content="https://agenciabrasil.ebc.com.br/educacao" />\n<meta name="twitter:title" content="Educação" />\n<meta itemprop="name" content="Educação" />\n  <title>Educação | Agência

In [177]:
sopa = BeautifulSoup(html, "html.parser")

In [178]:
sopa.title
# Agência Brasil
# UOL
# G1 Educação

<title>Educação | Agência Brasil</title>

In [20]:
sopa

<!DOCTYPE html>

<html dir="ltr" lang="pt-br" xmlns:fb="http://ogp.me/ns/fb#" xmlns:og="http://ogp.me/ns#">
<head>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<link href="https://agenciabrasil.ebc.com.br/sites/default/themes/agenciabrasil_v3/favicon.ico?thzdkc" rel="shortcut icon"/>
<link href="https://agenciabrasil.ebc.com.br/educacao" rel="canonical"/>
<meta content="Agência Brasil" property="og:site_name"/>
<meta content="article" property="og:type"/>
<meta content="https://agenciabrasil.ebc.com.br/educacao" property="og:url"/>
<meta content="Educação" property="og:title"/>
<meta content="summary" name="twitter:card"/>
<meta content="@agenciabrasil" name="twitter:site"/>
<meta content="@agenciabrasil" name="twitter:creator"/>
<meta content="https://agenciabrasil.ebc.com.br/educacao" name="twitter:url"/>
<meta content="Educação" name="twitter:title"/>
<meta content="Educação" itemprop="name"/>
<title>Educação | Agência Brasil</title>
<meta charset="utf-8"/>
<

In [179]:
url_ebc = 'https://agenciabrasil.ebc.com.br'
lista_links = []
links = sopa.find_all("a", href=True)
for link in links:
  if "/noticia/" not in link["href"]:
    continue

  link = f'{url_ebc}{link["href"]}'
  if link not in lista_links:
    lista_links.append(link)
lista_links

['https://agenciabrasil.ebc.com.br/educacao/noticia/2026-09/cartilha-reune-orientacoes-para-redacao-do-enem-2026',
 'https://agenciabrasil.ebc.com.br/educacao/noticia/2026-09/analfabetismo-entre-jovens-e-adultos-com-deficiencia-e-12-vezes-maior',
 'https://agenciabrasil.ebc.com.br/educacao/noticia/2026-09/pnd-2026-candidato-deve-responder-questionario-ate-sabado',
 'https://agenciabrasil.ebc.com.br/educacao/noticia/2026-09/escolas-da-bahia-e-do-para-vencem-premio-nacional-de-sustentabilidade',
 'https://agenciabrasil.ebc.com.br/educacao/noticia/2026-09/pnd-2026-tem-adesao-de-2007-municipios-e-23-estados',
 'https://agenciabrasil.ebc.com.br/educacao/noticia/2026-09/andes-lanca-documentario-sobre-importancia-da-educacao-e-ciencia',
 'https://agenciabrasil.ebc.com.br/educacao/noticia/2026-09/numero-de-professores-jovens-cai-31-em-dez-anos-aponta-pesquisa',
 'https://agenciabrasil.ebc.com.br/educacao/noticia/2026-09/prazo-para-recursos-do-enamed-2026-e-revalida-20262-vai-ate-quinta',
 'htt

In [302]:
lista_links, host = noticias(html)
host

'g1'

In [303]:
lista_links

['https://g1.globo.com/educacao/enem/2026/noticia/2026/09/18/inep-divulga-cartilha-de-redacao-do-enem-2026.ghtml',
 'https://g1.globo.com/educacao/noticia/2026/09/18/23percent-dos-jovens-com-dificuldade-de-mobilidade-e-sem-deficiencia-cognitiva-sao-analfabetos-no-brasil.ghtml',
 'https://g1.globo.com/educacao/noticia/2026/09/17/ensino-superior-faz-renda-subir-ate-110percent-veja-20-cursos-com-mais-formados-atuando-na-propria-area.ghtml',
 'https://g1.globo.com/educacao/noticia/2026/09/17/quanto-ganha-um-professor-no-brasil-veja-lista-que-mostra-salarios-a-partir-de-r-41-mil-e-entenda-deficit-com-outras-carreiras.ghtml',
 'https://g1.globo.com/educacao/noticia/2026/09/17/no-brasil-67percent-dos-alunos-mais-pobres-nao-atingem-nivel-minimo-de-leitura-entre-os-mais-ricos-26percent-estao-nesse-patamar.ghtml',
 'https://g1.globo.com/educacao/noticia/2026/09/16/apagao-numero-de-professores-idosos-mais-do-que-dobra-em-8-anos-enquanto-o-de-docentes-jovens-cai-31percent-diz-pesquisa.ghtml',
 'ht

In [223]:
def get_materia(link):
    partes = urlsplit(link)
    base = f"{partes.scheme}://{partes.hostname}"
    arquivo = partes.path.split("/")[-1]

    try:
        rp, delay = permissao(base)
        html = scraping(link, rp, delay)

    except PermissionError as erro:
        print(f'ERRO: {link} - {erro}')
        return None

    return html

In [311]:
link = 'https://agenciabrasil.ebc.com.br/educacao/noticia/2026-09/aulas-serao-suspensas-no-df-nas-sextas-feiras-antes-das-eleicoes'
html = get_materia(link)

In [318]:
sopa = BeautifulSoup(html, "html.parser")
titulo = sopa.find("h1")
if titulo is None:
    continue

titulo = titulo.get_text(" ", strip=True)


data = sopa.find("div", class_="data").text.strip()
data = re.search(r"(\d{2}/\d{2}/\d{4})\s*-\s*(\d{2}:\d{2})", data).group()
paragrafos = []
for i in sopa.find("div", class_="conteudo-noticia").find_all("p"):
  if len(i.get_text(' ', strip=True)) > 10:
    paragrafos.append(i.get_text(' ', strip=True))
texto = "\n".join(paragrafos)
palavras = len(texto.split())
print(titulo)
print(data)
print(texto)
print(palavras)

Aulas serão suspensas no DF nas sextas-feiras antes das eleições
11/09/2026 - 17:20
O governo do Distrito Federal (GDF) decidiu suspender as aulas nas redes de ensino público e privada nas duas sextas-feiras que antecedem o primeiro e o segundo turno das eleições deste ano. Com isso, não haverá aulas nos dias 2 de outubro e 23 de outubro.
Se não houver segundo turno, a medida será cancelada no dia 23 de outubro. O decreto do GDF foi publicado em edição extra do Diário Oficial.
A reposição de horas-aula na rede de ensino pública deve seguir as diretrizes da Secretaria de Estado de Educação. Nas instituições privadas, fica a critério de cada instituição de ensino.
Fica mantido o expediente dos responsáveis pela administração das instituições que funcionam como zonas eleitorais. Eles deverão se apresentar nas escolas a partir das 7h para receber as urnas eletrônicas que são distribuídas pela Justiça Eleitoral.
138


In [314]:
for i in sopa.find("div", class_="conteudo-noticia").find_all("p"):
  print(i.get_text(' ', strip=True))

O governo do Distrito Federal (GDF) decidiu suspender as aulas nas redes de ensino público e privada nas duas sextas-feiras que antecedem o primeiro e o segundo turno das eleições deste ano. Com isso, não haverá aulas nos dias 2 de outubro e 23 de outubro.
Se não houver segundo turno, a medida será cancelada no dia 23 de outubro. O decreto do GDF foi publicado em edição extra do Diário Oficial.
A reposição de horas-aula na rede de ensino pública deve seguir as diretrizes da Secretaria de Estado de Educação. Nas instituições privadas, fica a critério de cada instituição de ensino.
Fica mantido o expediente dos responsáveis pela administração das instituições que funcionam como zonas eleitorais. Eles deverão se apresentar nas escolas a partir das 7h para receber as urnas eletrônicas que são distribuídas pela Justiça Eleitoral.


teste_globo

In [300]:
rp, delay = permissao('https://g1.globo.com/educacao')

In [301]:
html = scraping('https://g1.globo.com/educacao', rp, delay)

In [55]:
sopa = BeautifulSoup(html, "html.parser")

In [33]:
sopa

<!DOCTYPE HTML>
<html class="" lang="pt-br"> <head><meta charset="utf-8"/><meta content="ie=edge,chrome=1" http-equiv="x-ua-compatible"/><meta content="width=device-width, initial-scale=1" name="viewport"/><script id="CDA_AAS" type="text/javascript">
  window.cdaaas = window.cdaaas || {};
  window.cdaaas.SETTINGS = window.cdaaas.SETTINGS || {};
  window.cdaaas.PAGE_ANALYTICS_DATA = {"channel":"desktop","contentId":"af1c3873-6f7f-4dca-8d2f-7f8154694f0e","contentType":"home","serviceWorker":true};
  window.cdaaas.helpers = window.cdaaas.helpers || {};
  window.cdaaas.internals = {};
  window.HorizonClient = new Promise((resolve, reject) => { 
    window.cdaaas.internals.resolveHorizonPromise = resolve; 
    window.cdaaas.internals.rejectHorizonPromise = reject; 
  });

  window.HorizonHelpers = {
    unloadCallbacks: [],
  };

  window.cdaaas.featureFlags = [];
  window.cdaaas.hasFF = (ff) => window.cdaaas.featureFlags.includes(ff);

  window.glbDebug = window.cdaaas.debugger = {
    fir

In [37]:
lista_links = []
links = sopa.find_all("a", href=True)
for link in links:
  endereco = link["href"]
  if ("/noticia/" not in endereco) or ("g1.globo.com/educacao/" not in endereco):
    continue
  if endereco not in lista_links:
    lista_links.append(endereco)
lista_links

['https://g1.globo.com/educacao/enem/2026/noticia/2026/09/18/inep-divulga-cartilha-de-redacao-do-enem-2026.ghtml',
 'https://g1.globo.com/educacao/noticia/2026/09/18/23percent-dos-jovens-com-dificuldade-de-mobilidade-e-sem-deficiencia-cognitiva-sao-analfabetos-no-brasil.ghtml',
 'https://g1.globo.com/educacao/noticia/2026/09/17/ensino-superior-faz-renda-subir-ate-110percent-veja-20-cursos-com-mais-formados-atuando-na-propria-area.ghtml',
 'https://g1.globo.com/educacao/noticia/2026/09/17/quanto-ganha-um-professor-no-brasil-veja-lista-que-mostra-salarios-a-partir-de-r-41-mil-e-entenda-deficit-com-outras-carreiras.ghtml',
 'https://g1.globo.com/educacao/noticia/2026/09/17/no-brasil-67percent-dos-alunos-mais-pobres-nao-atingem-nivel-minimo-de-leitura-entre-os-mais-ricos-26percent-estao-nesse-patamar.ghtml',
 'https://g1.globo.com/educacao/noticia/2026/09/16/apagao-numero-de-professores-idosos-mais-do-que-dobra-em-8-anos-enquanto-o-de-docentes-jovens-cai-31percent-diz-pesquisa.ghtml',
 'ht

In [280]:
link = 'https://g1.globo.com/educacao/noticia/2026/09/17/no-brasil-67percent-dos-alunos-mais-pobres-nao-atingem-nivel-minimo-de-leitura-entre-os-mais-ricos-26percent-estao-nesse-patamar.ghtml'
html = get_materia(link)

In [281]:
sopa = BeautifulSoup(html, "html.parser")
titulo = sopa.find("h1").text.strip()
data = sopa.find("p", class_="content-publication-data__updated").text.strip()
data = re.search(r"(\d{2}/\d{2}/\d{4})\s*(\d{2}h\d{2})", data).group().replace('h', ':').replace(' ', ' - ')
paragrafos = []
paragrafos.append(sopa.find("h2", class_='content-head__subtitle').text.strip())

for i in sopa.find_all("p", class_= 'content-text__container'):
  if len(i.get_text(' ', strip=True)) > 10:
    paragrafos.append(i.get_text(' ', strip=True))
texto = "\n".join(paragrafos)
palavras = len(texto.split())
print(titulo)
print(data)
print(texto)
print(palavras)

No Brasil, 67% dos alunos mais pobres não atingem nível mínimo de leitura; entre os mais ricos, 26% estão nesse patamar
17/09/2026 - 00:00
Análise do Instituto de Evidência Educativa, a partir de dados do Programa de Avaliação Internacional de Estudantes (Pisa) de 2025, mostrou que país tem a maior diferença no nível de leitura quando comparados alunos de maior e menor renda.
O Brasil é o país com a maior diferença no nível de leitura entre alunos de maior e menor nível socioeconômico na América Latina e no Caribe. Isso é o que aponta uma análise do Instituto de Evidência Educativa, a partir de dados do Programa de Avaliação Internacional de Estudantes (Pisa) de 2025.
➡️O levantamento mostrou que 67% dos alunos mais pobres no país não atingem o nível mínimo de leitura . Entre os mais ricos , 26% estão nesse patamar .
"Isso significa que a proporção de estudantes que não chega ao nível mínimo é 2,5 vezes maior no grupo de menor nível socioeconômico", destaca o relatório do instituto.
Pa

teste_uol

In [57]:
rp, delay = permissao('https://educacao.uol.com.br')

In [58]:
html = scraping('https://educacao.uol.com.br', rp, delay)

In [59]:
sopa = BeautifulSoup(html, "html.parser")
sopa

<!DOCTYPE html>
 <html lang="pt-br"> <head><meta charset="utf-8"/><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/><script>window.adsType = 'uolads';
        window.pushAds = window.pushAds || [];</script><script>var _comscore = _comscore || [];
        _comscore.push({
            c1: "2",
            c2: "6036356",
            options: {
                enableFirstPartyCookie: true,
                bypassUserConsentRequirementFor1PCookie: true
            }
        });
        (function () {
            var s = document.createElement("script"),
                el = document.getElementsByTagName("script")[0];
            s.async = true;
            s.src = "https://sb.scorecardresearch.com/cs/6036356/beacon.js";
            el.parentNode.insertBefore(s, el);
        })();</script> <title>UOL Educação: Pesquisa escolar, vestibular e dicas para alunos, pais e professores</title><link crossorigin="anonymous" href="https://stc.uol.com" rel="preconnect"/><link crossorigi

In [70]:
lista_links = []
links = sopa.find_all("a", href=True)
for link in links:
  endereco = link["href"]
  if ("/noticias/" not in endereco) or (("brasilescola" not in endereco) and ('educacao' not in endereco)):
    continue
  if endereco not in lista_links:
    lista_links.append(endereco)
lista_links

['https://educacao.uol.com.br/noticias/2026/09/18/pode-vir-ou-pode-vim-veja-qual-e-a-forma-correta-de-falar-e-escrever.ghtm',
 'https://educacao.uol.com.br/noticias/2026/09/17/falar-em-publico-exige-mais-do-que-dominio-do-assunto-veja-o-que-evitar.ghtm',
 'https://educacao.uol.com.br/noticias/2026/09/16/para-eu-ou-para-mim-veja-quando-usar-cada-expressao.ghtm',
 'https://educacao.uol.com.br/noticias/2026/09/16/enem-2026-o-que-muda-para-os-mais-de-5-milhoes-de-inscritos.ghtm',
 'https://educacao.uol.com.br/noticias/2026/09/18/quais-sao-as-diferencas-entre-mar-e-oceano.ghtm',
 'https://brasilescola.uol.com.br/noticias/uso-aplicativos-delivery-associado-sedentarismo-aponta-estudo/3133362.html',
 'https://vestibular.brasilescola.uol.com.br/noticias/ufrgs-vestibular-2027-inscricao-encerra-segunda/358901.html',
 'https://educador.brasilescola.uol.com.br/noticias/pnd-2026-acontece-domingo-veja-orientacoes/33540.html',
 'https://brasilescola.uol.com.br/noticias/enem-2026-faltam-20-dias-simulad

In [143]:
link = 'https://educacao.uol.com.br/noticias/2026/09/18/pode-vir-ou-pode-vim-veja-qual-e-a-forma-correta-de-falar-e-escrever.ghtm'
html = get_materia(link)

In [144]:
# https://educacao.uol.com.br/noticias
sopa = BeautifulSoup(html, "html.parser")
title = sopa.find("h1").text
data = sopa.find('time', class_='date').text
paragrafos = []
for i in sopa.find_all(class_='bullet'):
  paragrafos.append(i.get_text(' ', strip=True))
texto = "\n".join(paragrafos)
palavras = len(texto.split())
print(title)
print(data)
print(texto)
print(palavras)

Pode vir ou pode vim? Veja qual é a forma correta de falar e escrever
18/09/2026 05h30
"Pode vim aqui?" é uma frase bastante comum em conversas e mensagens. Na fala cotidiana, a construção pode soar natural, mas, na escrita de acordo com a norma padrão, a forma correta é "pode vir".
A confusão acontece porque "vir" e "vim" são formas do mesmo verbo, mas aparecem em situações diferentes. Para entender qual usar, basta observar o papel de cada uma na frase.
O correto é "pode vir"
Na expressão "pode vir", o verbo "poder" acompanha outro verbo e indica possibilidade ou permissão. Nessa construção, o segundo verbo permanece no infinitivo.
O infinitivo é a forma que encontramos no dicionário: vir.
Por isso, o correto é:
Você pode vir amanhã. Ela pode vir mais cedo. Pode vir quando quiser. O professor pode vir depois do intervalo.
A mesma estrutura aparece com diversos outros verbos:
Você pode sair agora. Ele pode fazer o trabalho amanhã. Nós podemos conversar depois.
Assim como não diríamos 

In [173]:
link = 'https://brasilescola.uol.com.br/noticias/enem-2026-faltam-20-dias-simulado-nacional-brasil-escola-gratuito-online/3133355.html'
html = get_materia(link)

In [174]:
# https://brasilescola.uol.com.br/noticias/
sopa = BeautifulSoup(html, "html.parser")
title = sopa.find("h1").text
data = sopa.find('p', class_='data-conteudo').text
data = re.search(r"(\d{2}/\d{2}/\d{4})\s*(\d{2}:\d{2})", data.strip().replace('h', ':')).group()
paragrafos = []
paragrafos.append(sopa.find('p', class_= 'resumo-artigo').text)
for i in sopa.find_all('p', class_= None):
  if len(i.get_text(' ', strip=True)) > 5:
    paragrafos.append(i.get_text(' ', strip=True))
texto = "\n".join(paragrafos)
palavras = len(texto.split())
print(title)
print(data)
print(texto)
print(palavras)

Enem 2026: faltam 20 dias para o Simulado Nacional Brasil Escola gratuito e online; veja como participar
18/09/2026 07:00
Provas do Simulado Nacional do Brasil Escola terão questões inéditas. Participe de forma gratuita.
Faltam 20 dias para o Simulado Nacional Brasil Escola - Enem 2026. As provas terão questões inéditas no modelo e formato do Exame Nacional do Ensino Médio (Enem).
Os participantes poderão fazer as provas entre os dias 8 e 22 de outubro , com participação totalmente gratuita e online.
Participe do Simulado Nacional Brasil Escola Enem 2026 aqui!
Será possível escolher qualquer horário e dia dentro do período em que o simulado estará aberto no site.
Dessa forma, o estudante pode se organizar melhor dentro da sua rotina e testar seus conhecimentos nas cinco áreas do conhecimento que fazem parte do Enem: Redação, Linguagens, Matemática, Ciências da Natureza e Ciências Humanas.
As questões e a proposta de redação são elaboradas pelo Statera Ensino Médio e Pré-Vestibular.
Lem

In [348]:
a = pd.read_csv(
    "/content/drive/MyDrive/"
    "Colab Notebooks/"
    "Bloco 5/"
    "Projeto de Bloco: Inteligência Artificial Aplicada [26E3_5]/"
    "noticias_19_09_2026.csv"
)

a

,titulo,data,texto,palavras,url,fonte
0,Inep divulga Cartilha de Redação do Enem 2026 ...,18/09/2026 - 11:08,"Reportagem do g1 mostrou que, em documento int...",905,https://g1.globo.com/educacao/enem/2026/notici...,g1 - Portal de notícias da Globo
1,23% dos jovens com dificuldade de mobilidade e...,18/09/2026 - 10:00,Levantamento do IBGE leva em conta brasileiros...,556,https://g1.globo.com/educacao/noticia/2026/09/...,g1 - Portal de notícias da Globo
2,Ensino superior faz renda subir até 110%; veja...,17/09/2026 - 15:00,"Levantamento aponta que 42,8% dos egressos cit...",1750,https://g1.globo.com/educacao/noticia/2026/09/...,g1 - Portal de notícias da Globo
3,Quanto ganha um professor no Brasil? Veja list...,17/09/2026 - 04:00,Pesquisa sobre risco de falta de professores m...,860,https://g1.globo.com/educacao/noticia/2026/09/...,g1 - Portal de notícias da Globo
4,"No Brasil, 67% dos alunos mais pobres não atin...",17/09/2026 - 00:00,"Análise do Instituto de Evidência Educativa, a...",599,https://g1.globo.com/educacao/noticia/2026/09/...,g1 - Portal de notícias da Globo
5,‘Apagão’: número de professores idosos mais do...,16/09/2026 - 15:00,Proximidade da aposentadoria para quantidade s...,753,https://g1.globo.com/educacao/noticia/2026/09/...,g1 - Portal de notícias da Globo
6,Dormir bem pode ajudar no desempenho escolar d...,16/09/2026 - 04:00,Pesquisadores analisaram como fatores como qua...,605,https://g1.globo.com/educacao/noticia/2026/09/...,g1 - Portal de notícias da Globo
7,O poder do recomeço: EJA transforma trajetória...,15/09/2026 - 12:58,Educação de Jovens e Adultos amplia oportunida...,944,https://g1.globo.com/educacao/especial-publici...,g1 - Portal de notícias da Globo
8,Cartilha reúne orientações para a redação do E...,18/09/2026 - 18:09,Os inscritos no Exame Nacional do Ensino Médio...,657,https://agenciabrasil.ebc.com.br/educacao/noti...,Agência Brasil - EBC
9,Analfabetismo entre jovens e adultos com defic...,18/09/2026 - 12:03,A taxa de analfabetismo entre as pessoas com d...,559,https://agenciabrasil.ebc.com.br/educacao/noti...,Agência Brasil - EBC
